# 🎤 super_Voz - StyleTTS2 One-Click Pipeline

Este notebook executa o treinamento completo do StyleTTS2 usando seus áudios do Google Drive.

### 🚀 Como usar:
1. Vá em **Ambiente de execução** -> **Alterar tipo de ambiente de execução** e selecione **GPU**.
2. Clique em **Ambiente de execução** -> **Executar tudo** (ou pressione `Ctrl + F9`).

**Estrutura recomendada no Drive:**
```text
MyDrive/super_Voz/
  Audios_brutos/ (seus áudios aqui)
```

In [ ]:
# @title 🛠️ Inicialização Automática
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys
import shutil

def setup_environment():
    print("--- 1. Montando Google Drive ---")
    if not Path('/content/drive').exists():
        drive.mount('/content/drive')
    else:
        print("Drive já montado.")

    REPO_URL = "https://github.com/warllemedicao/Voz_styllett2.git"
    REPO_BASE_DIR = Path("/content/Voz_styllett2")

    print(f"--- 2. Clonando/Atualizando Repositório: {REPO_URL} ---")
    if REPO_BASE_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_BASE_DIR), "pull"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_BASE_DIR)], check=True)

    # Busca recursiva pelo script principal para definir o PROJECT_DIR dinamicamente
    print("--- 3. Localizando diretório do projeto ---")
    target_script = "run_colab_styletts2.py"
    found_path = None
    for p in REPO_BASE_DIR.rglob(target_script):
        found_path = p.parent.parent # O script fica em scripts/, queremos a raiz do projeto
        break

    if not found_path or not found_path.exists():
        # Fallback se a busca falhar
        potential_dirs = [REPO_BASE_DIR / "super_Voz", REPO_BASE_DIR]
        for d in potential_dirs:
            if (d / "scripts" / target_script).exists():
                found_path = d
                break

    if not found_path:
        raise RuntimeError(f"Não foi possível localizar o script {target_script} dentro de {REPO_BASE_DIR}.")

    PROJECT_DIR = found_path.resolve()
    os.chdir(PROJECT_DIR)
    if str(PROJECT_DIR) not in sys.path:
        sys.path.append(str(PROJECT_DIR))

    print(f"--- 4. Instalando dependências críticas ---")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)

    print(f"\n✅ Ambiente Pronto!")
    print(f"Diretório do Projeto: {PROJECT_DIR}")
    return PROJECT_DIR

PROJECT_DIR = setup_environment()

In [ ]:
# @title 🚀 Iniciar Pipeline
import os
import subprocess
import sys
from pathlib import Path

# Garante que estamos no diretório certo
os.chdir(PROJECT_DIR)

cmd = [sys.executable, "-u", "scripts/run_colab_styletts2.py", "--config", "styletts2_colab_config.yml"]

print("="*60)
print("INICIANDO PIPELINE SUPER_VOZ")
print("="*60)
print(f"Comando: {' '.join(cmd)}")
print("Aguarde... as dependências serão verificadas antes do treino.\n")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")

code = proc.wait()
if code != 0:
    print(f"\n[ERRO] O pipeline falhou com código {code}.")
else:
    print("\n✅ Pipeline finalizado com sucesso!")